# Практическая работа 5.1. Генерация текста

Генерирование текста с помощью LSTM


В этой работе посмотрим, как можно использовать рекуррентные нейронные сети для генерирования последовательностей данных. В качестве примера будем генерировать текст, однако представленные здесь методы можно распространить на любые последовательные данные: вы можете применить их к последовательности музыкальных нот и получить новую музыку или к последовательности данных, описывающих мазки кистью (например, записанных в процессе рисования художником на iPad), и сгенерировать картину мазок за мазком и т. д.

Генерирование последовательностей данных не ограничивается созданием художественных произведений. Этот прием с успехом используется для синтеза речи и генерирования диалогов для чат-ботов. Функция Smart Reply, представленная компанией Google в 2016 году и способная автоматически генерировать короткие ответы на электронные письма или текстовые сообщения, основана на подобных приемах.

В конце 2014 года мало кто был знаком с аббревиатурой LSTM даже в сообществе машинного обучения. Успешное применение методов генерации последовательностей данных с помощью рекуррентных сетей начало приобретать широкую известность только в 2016 году. Но сами методы имеют довольно давнюю историю, начиная с разработки алгоритма LSTM в 1997 году1. Этот новый алгоритм первое время использовался для генерации текстов символ за символом.

В 2002 году Дуглас Эк (Douglas Eck), а затем и исследователи в швейцарской лаборатории им. Шмидхубера (Schmidhuber) применили алгоритм LSTM для генерации музыки и получили многообещающие результаты. В настоящее время Эк занимается исследованиями в подразделении Google Brain. В 2016 году он основал новую исследовательскую группу, получившую название Magenta, и сосредоточился на применении современных методов глубокого обучения для создания привлекательной музыки. Иногда хорошей идее требуется 15 лет, чтобы превратиться в осязаемый результат.

В конце 2000-х — начале 2010-х Алекс Грейвз (Alex Graves) проделал важную новаторскую работу по использованию рекуррентных сетей для генерации последовательностей данных. В частности, в 2013 году он работал над комбинацией рекуррентной и полносвязной сетей для получения человекоподобного почерка, используя временные последовательности позиций ручки, и эта работа расценивается некоторыми как поворотный момент1. Так, это конкретное применение нейронных сетей именно в этот момент времени открыло мне понятие мечтающих машин и подтолкнуло начать разработку фреймворка Keras. В 2013 году Грейвз оставил похожее закомментированное замечание, скрытое в файле LaTeX, выгруженном на сервер препринтов arXiv: «генерация последовательностей данных — это самая близкая к воплощению мечта компьютеров». Несколько лет спустя мы принимаем такие разработки как нечто само собой разумеющееся; однако в то время трудно было наблюдать за демонстрациями Грейвза и не приходить в восторг от открывающихся возможностей.

С тех пор рекуррентные сети с успехом используются для генерации музыки, диалогов, изображений, синтеза речи и проектирования молекул. Они даже использовались для создания сценария фильма, роли в котором исполняли живые актеры.


## Как генерируются последовательности данных?

Универсальный способ генерации последовательностей данных с применением методов глубокого обучения заключается в обучении сети (обычно рекуррентной или сверточной сети) для прогнозирования следующего токена или следующих нескольких токенов в последовательности, опираясь на предыдущие токены. Например, для входной последовательности «the cat is on the ma,» сеть обучается предсказывать целевую букву t2, следующий символ. Как обычно, при работе с текстовыми данными в роли токенов часто выступают слова или символы, и любая сеть, моделирующая вероятность появления следующего токена на основе предыдущих, называется языковой моделью. Языковая модель фиксирует скрытое пространство языка — его статистическую структуру.

После получения такой обученной языковой модели мы можем извлекать образцы из нее (генерировать новые последовательности): передать ей начальную строку текста (так называемые кондиционные данные), попросить сгенерировать следующий символ или слово (можно даже сгенерировать несколько слов сразу), добавить сгенерированный вывод в конец предыдущих входных данных и повторить процесс много раз. Этот цикл позволяет генерировать последовательности произвольной длины, отражающие структуру данных, на которых обучалась модель: последовательности, которые выглядят почти как предложения, написанные человеком. В примере, представленном далее в этом разделе, мы возьмем слой LSTM, передадим ему строки длиной N символов, извлеченные из текстового корпуса, и обучим его предсказывать символ N + 1. На выходе модель будет возвращать вектор softmax с вероятностями для всех возможных символов: распределение вероятностей для следующего символа. Такой слой LSTM называется языковой нейронной моделью уровня символов.

<div align="center">
  <img src="text_generation_process.png" alt="Процесс посимвольной генерации текста с использованием языковой модели" width="720"/>
  <br/>
  <em>Процесс посимвольной генерации текста с использованием языковой модели</em>
</div>


## Важность стратегии выбора

Для генерации текста важную роль играет алгоритм выбора следующего символа. Наивное решение — жадный выбор, когда выбирается наиболее вероятный символ. Но такой подход приводит к получению в результате повторяющихся, предсказуемых строк, которые не выглядят связными предложениями. Намного интереснее подход, который делает порой неожиданный выбор, вводя случайную составляющую в процесс выбора из распределения вероятностей для следующего символа. Этот подход называется стохастическим выбором (как вы помните, слово стохастический в данном контексте является синонимом слова случайный). Таким образом, если символ e имеет вероятность 0,3 стать следующим символом, согласно модели мы выберем его в 30 % случаев. Обратите внимание на то, что жадный выбор тоже может использоваться для выбора из распределения вероятностей, когда какой-то символ имеет вероятность 1, а все остальные — вероятность 0.

Вероятностный выбор из вектора softmax, возвращаемого моделью, является хорошим решением: он позволяет время от времени появляться в выводе даже маловероятным символам, генерировать более интересные предложения и иногда демонстрировать творческую жилку, придумывая новые, реалистично звучащие слова, которые отсутствуют в обучающих данных. Однако здесь есть одна проблема: эта стратегия не предусматривает возможности управлять величиной случайности в процессе выбора.

Зачем может понадобиться увеличивать или уменьшать случайную составляющую? Рассмотрим крайний случай: чисто случайный выбор, когда следующий символ выбирается из равномерно распределенных вероятностей и каждый символ одинаково вероятен. Эта схема имеет максимальную случайность; иными словами, это распределение вероятностей имеет максимальную энтропию. Естественно, она не произведет ничего интересного. С другой стороны, жадный выбор тоже не производит ничего интересного и не имеет случайной составляющей: соответствующее распределение вероятностей имеет минимальную энтропию. Выбор из «реального» распределения вероятностей — распределения, возвращаемого функцией softmax, — находится между этими двумя крайностями. Но есть еще множество других промежуточных точек с большей или меньшей энтропией, которые вы, возможно, захотите исследовать. Меньшая энтропия позволит генерировать последовательности с более предсказуемой структурой (и поэтому выглядящие более реалистичными), тогда как большая энтропия даст более неожиданный и творческий результат. Выбирая результаты из генеративных моделей, всегда полезно исследовать разные величины случайности в процессе генерации. Поскольку высшими судьями, определяющими, насколько интересны сгенерированные данные, являемся мы, люди, интересность оказывается весьма субъективной величиной, и поэтому нельзя сказать наперед, где лежит точка оптимальной энтропии.

Для управления величиной случайности в процессе выбора введем параметр, который назовем температурой softmax, характеризующий энтропию распределения вероятностей, используемую для выбора: она будет определять степень необычности или предсказуемости выбора следующего символа. С учетом значения temperature и на основе оригинального распределения вероятностей (результата функции softmax модели) будет вычисляться новое распределение путем взвешивания вероятностей, как показано ниже.

Взвешивание распределения вероятностей с учетомзначения температуры:


In [1]:
from __future__ import annotations

import os
import random
from pathlib import Path

PRACTICE_DIR = Path.cwd()

CACHE_DIR = PRACTICE_DIR / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(CACHE_DIR / "matplotlib")
os.environ["IPYTHONDIR"] = str(CACHE_DIR / "ipython")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

DATA_DIR = PRACTICE_DIR / ".data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["KERAS_HOME"] = str(DATA_DIR / "keras_home")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.config.set_visible_devices([], "GPU")


In [2]:
import numpy as np

def reweight_distribution(original_distribution, temperature=0.5): #original_distribution — это одномерный массив Numpy значений вероятностей, сумма которых должна быть равна 1
    """Возвращает взвешенную версию распределения вероятностей."""
    distribution = np.log(original_distribution) / temperature
    distribution = np.exp(distribution)
    return distribution / np.sum(distribution) #Возвращает новую, взвешенную версию оригинального распределения. Сумма вероятностей в новом распределении может получиться больше 1, поэтому разделим элементы вектора на сумму, чтобы получить новое распределение


Чем выше температура, тем больше энтропия распределения вероятностей и тем более неожиданными и менее структурированными будут генерируемые данные. Чем меньше температура, тем меньше будет величина случайной составляющей и тем более предсказуемыми будут генерируемые данные

<div align="center">
  <img src="temperature_sampling.png" alt="Разные результаты взвешивания одного и того же распределения вероятностей. Низкая температура = высокая предсказуемость, высокая температура = более случайный результат" width="720"/>
  <br/>
  <em>Разные результаты взвешивания одного и того же распределения вероятностей. Низкая температура = высокая предсказуемость, высокая температура = более случайный результат</em>
</div>


## Реализация посимвольной генерации текста на основе LSTM

Воплотим эти идеи на практике в реализации с Keras. Первое, что нам понадобится, — это много текстовых данных, на которых можно было бы обучить языковую модель. Для этого можно использовать любой большой текстовый файл или набор текстовых файлов, например статьи из Википедии, роман «Властелин колец» и т. д. В данном примере мы используем тексты из произведений Ницше, немецкого философа конца XIX века (в переводе на английский язык). Таким образом, в результате обучения у нас получится языковая модель, обладающая специфическими особенностями, характерными для произведений Ницше, а не обобщенная модель английского языка.

## Подготовка данных

Для начала загрузим корпус и преобразуем текст в нижний регистр.

Загрузка и парсинг исходного текстового файла


In [3]:
path = keras.utils.get_file(
    "nietzsche.txt",
    origin="https://s3.amazonaws.com/text-datasets/nietzsche.txt",
)
text = Path(path).read_text(encoding="utf-8").lower()
print("Corpus length:", len(text))


Corpus length: 600893


Затем извлечем частично перекрывающиеся последовательности с длиной maxlen, выполним прямое кодирование и упакуем в трехмерный массив Numpy x с формой (последовательности, максимальная_длина, уникальные_символы). Одновременно подготовим массив y с соответствующими целями: векторы с символами, полученные прямым кодированием, которые следуют за каждой извлеченной последовательностью.

Векторизация последовательностей символов


In [4]:
maxlen = 60 #Извлечение последовательностей по 60 символов
step = 3 #Новые последовательности выбираются через каждые 3 символа
sentences = [] #Хранение извлеченных последовательностей
next_chars = [] #Хранение целей (символов, следующих за последовательностями)
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))

chars = sorted(list(set(text))) #Список уникальных символов в корпусе
print("Unique characters:", len(chars))
char_indices = dict((char, chars.index(char)) for char in chars) # Словарь, отображающий уникальные символы в их индексы в списке «chars»

print("Vectorization...")
x = np.zeros((len(sentences), maxlen, len(chars)), dtype=bool)
y = np.zeros((len(sentences), len(chars)), dtype=bool)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1 #Прямое кодирование символов в бинарные массивы


Number of sequences: 200278
Unique characters: 57
Vectorization...


Конструирование сети

Эта сеть состоит из единственного слоя LSTM, за которым следует классификатор Dense с функцией softmax выбора из всех возможных символов. Но имейте в виду, что рекуррентные нейронные сети не единственный способ генерирования последовательностей данных; одномерные сверточные сети тоже показали превосходные результаты в решении этой задачи.

Модель с единственным слоем LSTM для предсказания следующего символа


In [5]:
model = keras.models.Sequential()
model.add(keras.Input(shape=(maxlen, len(chars))))
model.add(layers.LSTM(128))
model.add(layers.Dense(len(chars), activation="softmax"))
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        95,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 57)             │         7,353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,585 (400.72 KB)

 Trainable params: 102,585 (400.72 KB)

 Non-trainable params: 0 (0.00 B)

Так как цели имеют формат прямого кодирования, используем для обучения модели функцию потерь categorical_crossentropy.

Конфигурация компилируемой модели


In [6]:
optimizer = keras.optimizers.RMSprop(learning_rate=0.01)
model.compile(loss="categorical_crossentropy", optimizer=optimizer)


Обучение модели и извлечение образцов из нее

Имея обученную модель и фрагмент начального текста, можно сгенерировать новый текст, выполнив следующие пункты:

1. Извлечь из модели распределение вероятностей следующего символа для имеющегося на данный момент сгенерированного текста.

2. Выполнить взвешивание распределения с заданной температурой.

3. Выбрать следующий символ в соответствии с вновь взвешенным распределением вероятностей.

4. Добавить новый символ в конец текста.

Вот код, который мы используем для взвешивания оригинального распределения вероятностей, возвращаемого моделью, и извлечения индекса символа (функция выборки).

Функция выборки следующего символа с учетом прогнозов модели


In [7]:
def sample(preds, temperature=1.0):
    """Возвращает индекс следующего символа с учетом температуры softmax."""
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)


Наконец, следующий цикл повторяет обучение и генерирует текст. Для начала сгенерируем текст, использовав разные температуры после каждой эпохи. Это позволит вам увидеть, как меняется генерируемый тест по мере схождения модели и как температура влияет на стратегию выбора.

Цикл генерации текста


In [8]:
import random

output_path = PRACTICE_DIR / "СП-М-О-АЕЯМИИ-2025-Практика-5-1-БабушкинМВ.txt"

with output_path.open("w", encoding="utf-8") as output_file:
    for epoch in range(1, 60):    # Обучение модели в течение 60 эпох
        print(f"Эпоха {epoch}/59")
        history = model.fit(x, y, batch_size=128, epochs=1, verbose=1)
        print(f"loss: {history.history['loss'][-1]:.4f}")

        output_file.write(f"\n\nЭпоха {epoch}\n")

        # выбор случайного начального текста
        start_index = random.randint(0, len(text) - maxlen - 1)
        generated_text = text[start_index: start_index + maxlen]
        output_file.write('--- Generating with seed: "' + generated_text + '"\n')

        for temperature in [0.2, 0.5, 1.0, 1.2]:
            print("  temperature:", temperature)
            output_file.write(f"\n------ temperature: {temperature}\n")
            output_file.write(generated_text)

            # Генерация 400 символов, начиная с начального текста
            for i in range(400):
                sampled = np.zeros((1, maxlen, len(chars)))
                for t, char in enumerate(generated_text):
                    sampled[0, t, char_indices[char]] = 1.0

                preds = model.predict(sampled, verbose=0)[0]
                next_index = sample(preds, temperature)
                next_char = chars[next_index]

                generated_text += next_char
                generated_text = generated_text[1:]

                output_file.write(next_char)
            output_file.write("\n")
        output_file.flush()
        print()

print(f"Результаты генерации сохранены в файл: {output_path.name}")


Эпоха 1/59
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 74s 47ms/step - loss: 2.0257
loss: 2.0257
  temperature: 0.2
  temperature: 0.5
  temperature: 1.0
  temperature: 1.2

Эпоха 2/59
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 76s 49ms/step - loss: 1.6419
loss: 1.6419
  temperature: 0.2
  temperature: 0.5
  temperature: 1.0
  temperature: 1.2

Эпоха 3/59
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 77s 49ms/step - loss: 1.5365
loss: 1.5365
  temperature: 0.2
  temperature: 0.5
  temperature: 1.0
  temperature: 1.2

Эпоха 4/59
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 77s 49ms/step - loss: 1.4833
loss: 1.4833
  temperature: 0.2
  temperature: 0.5
  temperature: 1.0
  temperature: 1.2

Эпоха 5/59
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 78s 50ms/step - loss: 1.4513
loss: 1.4513
  temperature: 0.2
  temperature: 0.5
  temperature: 1.0
  temperature: 1.2

Эпоха 6/59
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 79s 50ms/step - loss: 1.4282
loss: 1.4282
  temperature: 0.2
  temperature: 0.5
  temperature: 1.0
  temperature: 1.2

Эпоха 7/59
1565/1565 ━━━━━━━━━━━━━

Как видите, при низком значении температуры генерируется часто повторяющийся и предсказуемый текст, однако локальная структура очень реалистична: в частности, все слова (слово является локальным шаблоном символов) — это действительные английские слова. При высоком значении температуры генерируется более интересный текст, неожиданный и даже творческий; в нем иногда появляются совершенно новые слова, кажущиеся правдоподобными (например, eterned и troveration). При высокой температуре внутренняя структура начинает разрушаться, и большинство слов выглядят как полуслучайные последовательности символов. Без сомнения, 0,5 — самая интересная температура для генерации текста в данном конкретном решении. Всегда пробуйте несколько стратегий выбора! Разумный баланс между изученной структурой и случайностью — вот что делает сгенерированные данные интересными.

Обратите внимание на то, что, обучая модель большего размера дольше и на большем объеме данных, можно добиться генерации текста, который выглядит еще реалистичнее. Однако не думайте, что когда-нибудь вам удастся сгенерировать осмысленный текст, разве только по чистой случайности: вы всего лишь выбираете образцы данных из статистической модели, в которой символы следуют за символами. Язык — это канал общения, и существуют различия между сутью общения и статистической структурой сообщений, которыми кодируется общение. Чтобы осознать это различие, проведите мысленный эксперимент: представьте, что человеческий язык позволял бы сжимать информацию при общении почти так же, как это делают компьютеры с цифровой информацией. Язык не потерял бы своей осмысленности, но утратил статистическую структуру, что сделало бы невозможным обучение языковой модели, как мы только что это проделали.
